In [13]:
# import libraries
from google.colab import files
import pandas as pd
import numpy as np
from sklearn import metrics
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

In [14]:
# pycaret
!pip install pycaret
import pycaret as pc

In [15]:
from pycaret.classification import *

In [16]:
#from pc.utils import enable_colab
#enable_colab()

# ERROR IN ABOVE - BUT NOT NECESSARY FOR MODELING

In [18]:
# load Cleveland datafile

##--original
#cleveland_csv_path = "/processed.cleveland.data"

##--modified
cleveland_csv_path = "processed.cleveland.data"

cleveland_data = pd.read_csv(cleveland_csv_path, header = None)
# assign column names
##--original
# cleveland_data.set_axis(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
#                   'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
#                   'ca', 'thal', 'num'], axis = 1, inplace = True)

##--modified
cleveland_data.columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
    'ca', 'thal', 'num'
]

In [19]:
# Define label column
heart_label = 'num'

# SLIGHTLY BETTER MODEL
no missing values & stratified sampling

## DATA PREPROCESSING

### MISSING VALUES

In [20]:
# make copy of original dataframe
cleveland = cleveland_data.copy()

In [21]:
# check for missing values
cleveland.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    object 
 12  thal      303 non-null    object 
 13  num       303 non-null    int64  
dtypes: float64(11), int64(1), object(2)
memory usage: 33.3+ KB


In [22]:
# replace '?' values with NaN so can impute
cleveland['thal'].replace('?', np.NaN, inplace=True)
cleveland['thal'] = cleveland['thal'].astype(float)
cleveland['ca'].replace('?', np.NaN, inplace=True)
cleveland['ca'] = cleveland['ca'].astype(float)
cleveland.tail() # to verify didn't mess up dataframe IDs

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1
302,38.0,1.0,3.0,138.0,175.0,0.0,0.0,173.0,0.0,0.0,1.0,NaN,3.0,0


In [23]:
from sklearn.impute import SimpleImputer
# impute with mode as 'thal' and 'ca' (attributes w/ missing values) are discrete
imputeMode = SimpleImputer(strategy="most_frequent") # create mode imputer
imputeMode.fit(cleveland) # fit - learns the data
imputed = imputeMode.transform(cleveland) # transform - imputes with chosen strategy
cleveland = pd.DataFrame(imputed, columns=cleveland.columns, index=cleveland['thal'].index) # back to pandas DataFrame
cleveland.info() # check for missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
 13  num       303 non-null    float64
dtypes: float64(14)
memory usage: 33.3 KB


In [24]:
cleveland.head() # check for anything obviously wonky

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0.0


In [25]:
cleveland.corr()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
age,1.000000,-0.097542,0.104139,0.284946,0.208950,0.118530,0.148868,-0.393806,0.091661,0.203805,0.161770,0.365323,0.128303,0.222853
sex,-0.097542,1.000000,0.010084,-0.064456,-0.199915,0.047862,0.021647,-0.048663,0.146201,0.102173,0.037533,0.086048,0.380581,0.224469
cp,0.104139,0.010084,1.000000,-0.036077,0.072319,-0.039975,0.067505,-0.334422,0.384060,0.202277,0.152050,0.233117,0.262089,0.407075
trestbps,0.284946,-0.064456,-0.036077,1.000000,0.130120,0.175340,0.146560,-0.045351,0.064762,0.189171,0.117382,0.097528,0.134424,0.157754
chol,0.208950,-0.199915,0.072319,0.130120,1.000000,0.009841,0.171043,-0.003432,0.061310,0.046564,-0.004062,0.123726,0.018351,0.070909
fbs,0.118530,0.047862,-0.039975,0.175340,0.009841,1.000000,0.069564,-0.007854,0.025665,0.005747,0.059894,0.140764,0.064625,0.059186
restecg,0.148868,0.021647,0.067505,0.146560,0.171043,0.069564,1.000000,-0.083389,0.084867,0.114133,0.133946,0.131749,0.024325,0.183696
thalach,-0.393806,-0.048663,-0.334422,-0.045351,-0.003432,-0.007854,-0.083389,1.000000,-0.378103,-0.343085,-0.385601,-0.265699,-0.274142,-0.415040
exang,0.091661,0.146201,0.384060,0.064762,0.061310,0.025665,0.084867,-0.378103,1.000000,0.288223,0.257748,0.145788,0.325240,0.397057
oldpeak,0.203805,0.102173,0.202277,0.189171,0.046564,0.005747,0.114133,-0.343085,0.288223,1.000000,0.577537,0.301067,0.342405,0.504092


In [26]:
cleveland.drop(['slope', 'cp', 'thal', 'fbs', 'restecg'], axis=1, inplace=True)

### BINARY LABEL
(multiclass for "future research")

In [27]:
cleveland['num'][cleveland['num'] > 0] = 1
cleveland.head()

,age,sex,trestbps,chol,thalach,exang,oldpeak,ca,num
0,63.0,1.0,145.0,233.0,150.0,0.0,2.3,0.0,0.0
1,67.0,1.0,160.0,286.0,108.0,1.0,1.5,3.0,1.0
2,67.0,1.0,120.0,229.0,129.0,1.0,2.6,2.0,1.0
3,37.0,1.0,130.0,250.0,187.0,0.0,3.5,0.0,0.0
4,41.0,0.0,130.0,204.0,172.0,0.0,1.4,0.0,0.0


### STRATIFIED SAMPLING

(due to difference sex representation)

In [28]:
cleveland_strat = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in cleveland_strat.split(cleveland, cleveland["num"]):
  cleveland_strat_train = cleveland.loc[train_index]
  cleveland_strat_test = cleveland.loc[test_index]

In [29]:
cleveland_strat_test["num"].value_counts()/len(cleveland_strat_test)

,count
num,
0.0,0.540984
1.0,0.459016


In [30]:
cleveland_strat_train["num"].value_counts()/len(cleveland_strat_train)

,count
num,
0.0,0.541322
1.0,0.458678


Less significant for Cleveland and Statlog datasets, which have less dramatic difference in sex representation in data, but different for other three datasets.

In [31]:
# split stratified data
cleveland_strat_train_X = cleveland_strat_train.drop([heart_label], axis=1)
cleveland_strat_train_y = cleveland_strat_train[heart_label]
cleveland_strat_test_X = cleveland_strat_test.drop([heart_label], axis=1)
cleveland_strat_test_y = cleveland_strat_test[heart_label]
cleveland_strat_train_X.sample(5)

,age,sex,trestbps,chol,thalach,exang,oldpeak,ca
299,68.0,1.0,144.0,193.0,141.0,0.0,3.4,2.0
240,41.0,1.0,110.0,235.0,153.0,0.0,0.0,0.0
191,51.0,1.0,140.0,298.0,122.0,1.0,4.2,3.0
45,58.0,1.0,112.0,230.0,165.0,0.0,2.5,1.0
150,52.0,1.0,152.0,298.0,178.0,0.0,1.2,0.0


## Correlated Block

In [32]:
# set up for correlated
cleveland_pc_setup = setup(data = cleveland_strat_train, test_data = cleveland_strat_test, target = 'num', session_id=42)

,Description,Value
0,Session id,42
1,Target,num
2,Target type,Binary
3,Original data shape,"(303, 9)"
4,Transformed data shape,"(303, 9)"
5,Transformed train set shape,"(242, 9)"
6,Transformed test set shape,"(61, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


In [33]:
# compare models
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7975,0.8724,0.7402,0.8172,0.7681,0.5901,0.6008,0.9390
ridge,Ridge Classifier,0.7892,0.8735,0.7038,0.8248,0.7506,0.5712,0.5850,0.0240
et,Extra Trees Classifier,0.7848,0.8643,0.7576,0.7778,0.7637,0.5668,0.5715,0.2160
lda,Linear Discriminant Analysis,0.7808,0.8735,0.6947,0.8162,0.7427,0.5547,0.5674,0.0360
nb,Naive Bayes,0.7767,0.8651,0.7303,0.7935,0.7524,0.5500,0.5603,0.0240
rf,Random Forest Classifier,0.7720,0.8374,0.7023,0.7820,0.7360,0.5371,0.5435,0.1850
qda,Quadratic Discriminant Analysis,0.7642,0.8493,0.7030,0.7770,0.7330,0.5230,0.5299,0.0260
gbc,Gradient Boosting Classifier,0.7598,0.8327,0.7295,0.7430,0.7338,0.5155,0.5183,0.1790
lightgbm,Light Gradient Boosting Machine,0.7557,0.8322,0.6939,0.7652,0.7224,0.5056,0.5135,0.0800
xgboost,Extreme Gradient Boosting,0.7473,0.8118,0.7114,0.7457,0.7202,0.4905,0.5000,0.0670


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=42, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [34]:
# create model
lr = create_model('lr')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7600,0.8654,0.5833,0.8750,0.7000,0.5130,0.5424
1,0.8400,0.9286,0.8182,0.8182,0.8182,0.6753,0.6753
2,0.6667,0.6993,0.6364,0.6364,0.6364,0.3287,0.3287
3,0.8750,0.8601,0.8182,0.9000,0.8571,0.7465,0.7492
4,0.8750,0.9580,0.8182,0.9000,0.8571,0.7465,0.7492
5,0.7500,0.8601,0.7273,0.7273,0.7273,0.4965,0.4965
6,0.7917,0.9720,0.5455,1.0000,0.7059,0.5652,0.6276
7,0.8333,0.8671,0.9091,0.7692,0.8333,0.6690,0.6783
8,0.8333,0.8811,0.8182,0.8182,0.8182,0.6643,0.6643


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [35]:
# tune model
tuned_lr = tune_model(lr)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8400,0.8462,0.7500,0.9000,0.8182,0.6774,0.6864
1,0.8400,0.9351,0.8182,0.8182,0.8182,0.6753,0.6753
2,0.6250,0.6923,0.6364,0.5833,0.6087,0.2500,0.2509
3,0.8750,0.8531,0.8182,0.9000,0.8571,0.7465,0.7492
4,0.8750,0.9510,0.8182,0.9000,0.8571,0.7465,0.7492
5,0.7500,0.8671,0.7273,0.7273,0.7273,0.4965,0.4965
6,0.8333,0.9650,0.6364,1.0000,0.7778,0.6547,0.6976
7,0.8333,0.8741,0.9091,0.7692,0.8333,0.6690,0.6783
8,0.8333,0.8741,0.8182,0.8182,0.8182,0.6643,0.6643


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [36]:
predict_model(tuned_lr)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.8525,0.9383,0.8929,0.8065,0.8475,0.7053,0.7087


,age,sex,trestbps,chol,thalach,exang,oldpeak,ca,num,prediction_label,prediction_score
219,59.0,1.0,138.0,271.0,182.0,0.0,0.0,0.0,0.0,0,0.8460
271,66.0,1.0,160.0,228.0,138.0,0.0,2.3,0.0,0.0,1,0.5899
89,51.0,0.0,130.0,256.0,149.0,0.0,0.5,0.0,0.0,0,0.9074
101,34.0,1.0,118.0,182.0,174.0,0.0,0.0,0.0,0.0,0,0.8645
67,54.0,1.0,150.0,232.0,165.0,0.0,1.6,0.0,0.0,0,0.6503
...,...,...,...,...,...,...,...,...,...,...,...
285,58.0,1.0,114.0,318.0,140.0,0.0,4.4,3.0,1.0,1,0.9892
243,61.0,1.0,134.0,234.0,145.0,0.0,2.6,2.0,1.0,1,0.9151
94,63.0,0.0,135.0,252.0,172.0,0.0,0.0,0.0,0.0,0,0.9643
291,55.0,0.0,132.0,342.0,166.0,0.0,1.2,0.0,0.0,0,0.8946


## BINARY MODEL & Uncorreleated (do not execute to keep printed..)

In [37]:
# set up
cleveland_pc_setup = setup(data = cleveland_strat_train, test_data = cleveland_strat_test, target = 'num', session_id=42)

,Description,Value
0,Session id,42
1,Target,num
2,Target type,Binary
3,Original data shape,"(303, 9)"
4,Transformed data shape,"(303, 9)"
5,Transformed train set shape,"(242, 9)"
6,Transformed test set shape,"(61, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


In [38]:
# compare models
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7975,0.8724,0.7402,0.8172,0.7681,0.5901,0.6008,0.0580
ridge,Ridge Classifier,0.7892,0.8735,0.7038,0.8248,0.7506,0.5712,0.5850,0.0270
et,Extra Trees Classifier,0.7848,0.8643,0.7576,0.7778,0.7637,0.5668,0.5715,0.2010
lda,Linear Discriminant Analysis,0.7808,0.8735,0.6947,0.8162,0.7427,0.5547,0.5674,0.0440
nb,Naive Bayes,0.7767,0.8651,0.7303,0.7935,0.7524,0.5500,0.5603,0.0240
rf,Random Forest Classifier,0.7720,0.8374,0.7023,0.7820,0.7360,0.5371,0.5435,0.1870
qda,Quadratic Discriminant Analysis,0.7642,0.8493,0.7030,0.7770,0.7330,0.5230,0.5299,0.0260
gbc,Gradient Boosting Classifier,0.7598,0.8327,0.7295,0.7430,0.7338,0.5155,0.5183,0.1630
lightgbm,Light Gradient Boosting Machine,0.7557,0.8322,0.6939,0.7652,0.7224,0.5056,0.5135,0.1750
xgboost,Extreme Gradient Boosting,0.7473,0.8118,0.7114,0.7457,0.7202,0.4905,0.5000,0.0520


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=42, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [39]:
# create model
lda = create_model('lda')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7600,0.8910,0.5833,0.8750,0.7000,0.5130,0.5424
1,0.8400,0.9416,0.8182,0.8182,0.8182,0.6753,0.6753
2,0.6250,0.6993,0.5455,0.6000,0.5714,0.2394,0.2403
3,0.8333,0.8671,0.7273,0.8889,0.8000,0.6596,0.6693
4,0.7917,0.9580,0.6364,0.8750,0.7368,0.5714,0.5913
5,0.7500,0.8322,0.7273,0.7273,0.7273,0.4965,0.4965
6,0.7917,0.9720,0.5455,1.0000,0.7059,0.5652,0.6276
7,0.7917,0.8601,0.8182,0.7500,0.7826,0.5833,0.5854
8,0.8750,0.8671,0.8182,0.9000,0.8571,0.7465,0.7492


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [40]:
# tune model
tuned_lda = tune_model(lda)

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7600,0.9038,0.5833,0.8750,0.7000,0.5130,0.5424
1,0.8400,0.9286,0.8182,0.8182,0.8182,0.6753,0.6753
2,0.6667,0.6783,0.5455,0.6667,0.6000,0.3191,0.3239
3,0.8333,0.8741,0.7273,0.8889,0.8000,0.6596,0.6693
4,0.7917,0.9650,0.6364,0.8750,0.7368,0.5714,0.5913
5,0.7500,0.8182,0.7273,0.7273,0.7273,0.4965,0.4965
6,0.8333,0.9510,0.6364,1.0000,0.7778,0.6547,0.6976
7,0.8750,0.8741,0.9091,0.8333,0.8696,0.7500,0.7526
8,0.8750,0.8811,0.8182,0.9000,0.8571,0.7465,0.7492


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [41]:
# display tuned model
print(tuned_lda)

LinearDiscriminantAnalysis(covariance_estimator=None, n_components=None,
                           priors=None, shrinkage='auto', solver='lsqr',
                           store_covariance=False, tol=0.0001)


In [42]:
predict_model(tuned_lda)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Linear Discriminant Analysis,0.8361,0.9459,0.8571,0.8000,0.8276,0.6717,0.6731


,age,sex,trestbps,chol,thalach,exang,oldpeak,ca,num,prediction_label,prediction_score
219,59.0,1.0,138.0,271.0,182.0,0.0,0.0,0.0,0.0,0,0.9186
271,66.0,1.0,160.0,228.0,138.0,0.0,2.3,0.0,0.0,1,0.5986
89,51.0,0.0,130.0,256.0,149.0,0.0,0.5,0.0,0.0,0,0.9532
101,34.0,1.0,118.0,182.0,174.0,0.0,0.0,0.0,0.0,0,0.9496
67,54.0,1.0,150.0,232.0,165.0,0.0,1.6,0.0,0.0,0,0.7410
...,...,...,...,...,...,...,...,...,...,...,...
285,58.0,1.0,114.0,318.0,140.0,0.0,4.4,3.0,1.0,1,0.9914
243,61.0,1.0,134.0,234.0,145.0,0.0,2.6,2.0,1.0,1,0.9180
94,63.0,0.0,135.0,252.0,172.0,0.0,0.0,0.0,0.0,0,0.9789
291,55.0,0.0,132.0,342.0,166.0,0.0,1.2,0.0,0.0,0,0.9447
